# 18_01_列で分けて_行で分けて_1_回だけ足す

コード：`ch18/blocks/18_01_列で分けて_行で分けて_1_回だけ足す.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 18 章：Megatron-LM 型のテンソル並列。MLP の 1 つ目の行列を列で、2 つ目を行で分ける
import numpy as np

rng = np.random.default_rng(0)
tokens, d, d_ff, n_gpu = 8, 64, 256, 4
X = rng.normal(size=(tokens, d))
A = rng.normal(size=(d, d_ff)) / np.sqrt(d)        # 1 つ目の重み（d → d_ff）
B = rng.normal(size=(d_ff, d)) / np.sqrt(d_ff)     # 2 つ目の重み（d_ff → d）
gelu = lambda z: 0.5 * z * (1 + np.tanh(np.sqrt(2 / np.pi) * (z + 0.044715 * z ** 3)))

full = gelu(X @ A) @ B                             # 1 台で計算した答え

# 列で分けた A_i と、行で分けた B_i を GPU i が持つ
A_parts = np.split(A, n_gpu, axis=1)               # 各 (d, d_ff/n)
B_parts = np.split(B, n_gpu, axis=0)               # 各 (d_ff/n, d)
partial = [gelu(X @ Ai) @ Bi for Ai, Bi in zip(A_parts, B_parts)]   # GPU ごとに独立に計算
tp = sum(partial)                                  # ★ ここだけが通信（all-reduce で足し合わせる）
print(f"列→行の分割：1 台の答えとの最大差 {np.abs(tp - full).max():.1e}")

# 逆に、1 つ目を行で分けると GeLU の前で足し合わせが要り、通信が 1 回増える
A_rows = np.split(A, n_gpu, axis=0); X_cols = np.split(X, n_gpu, axis=1)
wrong = sum(gelu(Xi @ Ai) for Xi, Ai in zip(X_cols, A_rows)) @ B   # GeLU の後で足すと…
print(f"1 つ目を行で分け、GeLU の後で足した場合の最大差 {np.abs(wrong - full).max():.1e}（非線形の前に集める必要がある）")

# GPU 1 台あたりの重みと、1 層あたりの通信量
per_gpu = A_parts[0].size + B_parts[0].size
print(f"重みの要素数：1 台 {A.size + B.size:,} → GPU 1 台あたり {per_gpu:,}（1/{n_gpu}）")
print(f"順伝播の通信：1 層の MLP あたり all-reduce 1 回、量は tokens×d = {tokens * d:,} 要素")
